## Step 02 — Transcribe videos with Whisper

This notebook scans the downloaded video folder and transcribes every video file using [faster-whisper](https://github.com/guillaumekln/faster-whisper), an optimised re-implementation of [OpenAI Whisper](https://github.com/openai/whisper) that runs significantly faster with the same accuracy.

For each video it:

1. Extracts its audio in the form of a WAV file using `ffmpeg` and saves it to a folder
2. Runs Whisper on the cached audio to produce a transcript and detect the spoken language
3. Saves results to a CSV, checkpointing every `SAVE_EVERY` files so progress is not lost if the process is interrupted

Transcription runs across multiple threads to process several videos in parallel.

**Output:** `output/transcriptions.csv`: one row per video with columns `file_name`, `transcript`, `language`, and `path`.

Libraries used:

- `faster-whisper` — speech-to-text transcription
- `torch` — detects whether a CUDA GPU is available so the model uses it automatically
- `concurrent.futures.ThreadPoolExecutor` — parallel processing across multiple files
- `tqdm` — progress bar
- `ffmpeg` (system dependency, not a Python package) — audio extraction

**NOTE:** Before running locally, install the `ffmpeg` system dependency by following the [ffmpeg install guide](https://github.com/oop7/ffmpeg-install-guide).


In [ ]:
# INSTALL REQUIREMENTS (one time only):

# (locally)
# %pip install -r ../requirements.txt

# (extra step for local setup)
# Install ffmpeg via this guide: (https://github.com/oop7/ffmpeg-install-guide)

# (on google colab)
# !pip install -q faster-whisper
# !apt-get -qq install ffmpeg

import os
import glob
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from faster_whisper import WhisperModel

# (on google colab)
# from google.colab import drive
# drive.mount('/content/drive')


### Transcription

The cell below handles everything: loading files, detecting the device, loading the Whisper model, resuming from a previous run if the output CSV already exists, and running transcription in parallel threads. Scroll through the section headers inside the cell for a step-by-step breakdown.


In [ ]:
# =============================
# 1. SET TRANSCRIPTION CONFIGS
# =============================

# In google drive: 
# PATH_PREFIX = "/content/drive/MyDrive/tiktok-content-reupload-tracker"

# Locally:
PATH_PREFIX = ".."

BASE_DIR = f"{PATH_PREFIX}/output/videofiles-extracted/"
AUDIO_CACHE = f"{PATH_PREFIX}/output/audio-cache/"
OUTPUT_CSV = f"{PATH_PREFIX}/output/transcriptions.csv"

MODEL_SIZE = "base"
NUM_WORKERS = 2 
SAVE_EVERY = 10

os.makedirs(AUDIO_CACHE, exist_ok=True)

# =============================
# 2. LOAD FILES TO TRANSCRIBE
# =============================
video_exts = ["mp4", "mov", "mkv", "webm"]
audio_exts = ["mp3", "wav", "m4a", "aac"]

files = []
for ext in video_exts + audio_exts:
    files.extend(glob.glob(BASE_DIR + f"*.{ext}"))

print(f"Found {len(files)} media files")

# =============================
# 3. LOAD TRANSCRIPTION MODEL
# =============================

device = "cuda" if torch.cuda.is_available() else "cpu"

compute_type = "float16" if device == "cuda" else "int8"  # important

model = WhisperModel(
    MODEL_SIZE,
    device=device,
    compute_type=compute_type
)

# =============================
# 4. RESUME TRANSCRIPTION FROM LAST SAVED 
# =============================
if os.path.exists(OUTPUT_CSV):
    df_existing = pd.read_csv(OUTPUT_CSV)
    done_files = set(df_existing["file_name"].tolist())
    rows = df_existing.to_dict("records")
    print(f"Resuming: {len(done_files)} already done")
else:
    rows = []
    done_files = set()

# =============================
# 5. CACHE EXTRACTED AUDIO 
# =============================
def extract_audio(path):
    file_name = Path(path).stem
    out_path = os.path.join(AUDIO_CACHE, file_name + ".wav")

    if os.path.exists(out_path):
        return out_path  # reuse cached audio

    os.system(
        f'ffmpeg -loglevel error -y -i "{path}" -ar 16000 -ac 1 "{out_path}"'
    )
    return out_path

# =============================
# 6. DEFINE TRANSCRIPTION FUNCTION
# =============================
def process_file(path):
    file_name = Path(path).name

    if file_name in done_files:
        return None

    try:
        audio_path = extract_audio(path)

        segments, info = model.transcribe(
            audio_path,
            beam_size=1
        )

        text = " ".join([seg.text for seg in segments])

        return {
            "file_name": file_name,
            "transcript": text.strip(),
            "language": info.language,
            "path": path
        }

    except Exception as e:
        print(f"Failed: {file_name} -> {e}")
        return None

# =============================
# 7. RUN TRANSCRIPTION VIA PARALLEL PROCESSING
# =============================
print("Starting transcription...")

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    for i, result in enumerate(
        tqdm(executor.map(process_file, files), total=len(files))
    ):
        if result:
            rows.append(result)

        if i % SAVE_EVERY == 0 and i > 0:
            pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

# =============================
# 8. SAVE RESULTS
# =============================
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print("✅ Done. Saved to:", OUTPUT_CSV)